# Plant Disease Classification: MobileNetV2 vs EfficientNet-B0

Training, validation, checkpointing, and evaluation code. Dataset loading is intentionally left as a placeholder.

Set `NUM_CLASSES` and `CLASS_NAMES` to match your dataset. Both models should use the same underlying split assignments and class-to-index mapping, but their own pretrained preprocessing.

## 1. Imports and configuration

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from sklearn.metrics import accuracy_score, f1_score, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 3
NUM_EPOCHS = 10

# Replace these example names with your real classes in dataset-index order.
CLASS_NAMES = ["healthy", "disease_a", "disease_b"]

print("Using device:", device)

## 2. Load pretrained models

In [ ]:
weights_mnv2 = models.MobileNet_V2_Weights.DEFAULT
model_mnv2 = models.mobilenet_v2(weights=weights_mnv2)

weights_eff = models.EfficientNet_B0_Weights.DEFAULT
model_eff = models.efficientnet_b0(weights=weights_eff)

## 3. Replace the final classifiers

In [ ]:
model_mnv2.classifier[1] = nn.Linear(
    model_mnv2.classifier[1].in_features, NUM_CLASSES
)
model_eff.classifier[1] = nn.Linear(
    model_eff.classifier[1].in_features, NUM_CLASSES
)

## 4. Freeze backbones and enable classifier training

In [ ]:
for param in model_mnv2.parameters():
    param.requires_grad = False
for param in model_mnv2.classifier.parameters():
    param.requires_grad = True

for param in model_eff.parameters():
    param.requires_grad = False
for param in model_eff.classifier.parameters():
    param.requires_grad = True

model_mnv2 = model_mnv2.to(device)
model_eff = model_eff.to(device)

## 5. Optimizers and loss

In [ ]:
optimizer_mnv2 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_mnv2.parameters()), lr=0.001
)
optimizer_eff = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_eff.parameters()), lr=0.001
)
criterion = nn.CrossEntropyLoss()

## 6. Preprocessing and dataset placeholder

Create these six loaders after adding your dataset:
- `train_loader_mnv2`, `val_loader_mnv2`, `test_loader_mnv2`
- `train_loader_eff`, `val_loader_eff`, `test_loader_eff`

Each loader yields `(images, labels)`. Use random augmentation only for training; validation and test transforms should be deterministic.

In [ ]:
preprocess_mnv2 = weights_mnv2.transforms()
preprocess_eff = weights_eff.transforms()

# TODO: Create datasets and DataLoaders here.
# Keep underlying split assignments and class indices identical for both models.
# Apply each model's own preprocessing to its corresponding dataset.

## 7. Train for one epoch

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    total_images = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_images += batch_size

    if total_images == 0:
        raise ValueError("The training loader yielded no images.")

    return total_loss / total_images

## 8. Collect predictions

In [ ]:
def get_predictions(model, data_loader):
    model.eval()
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            all_labels.extend(labels.cpu().tolist())
            all_predictions.extend(predictions.cpu().tolist())

    return all_labels, all_predictions

## 9. Train and save the best macro-F1 checkpoint

In [ ]:
def train_model(
    model, train_loader, val_loader, optimizer, criterion,
    checkpoint_path, num_epochs
):
    best_val_f1 = -1.0

    for epoch in range(num_epochs):
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer
        )
        true_labels, predictions = get_predictions(model, val_loader)
        val_f1 = f1_score(
            true_labels, predictions, average="macro", zero_division=0
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), checkpoint_path)

        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Macro F1: {val_f1:.4f} | Best Val F1: {best_val_f1:.4f}"
        )

    model.load_state_dict(torch.load(
        checkpoint_path, map_location=device, weights_only=True
    ))
    return model

## 10. Train both models
Run this section after creating all six DataLoaders in Section 6.

In [ ]:
# Uncomment after defining the dataset loaders.

# model_mnv2 = train_model(
#     model_mnv2, train_loader_mnv2, val_loader_mnv2,
#     optimizer_mnv2, criterion, "best_mobilenet.pth", NUM_EPOCHS
# )

# model_eff = train_model(
#     model_eff, train_loader_eff, val_loader_eff,
#     optimizer_eff, criterion, "best_efficientnet.pth", NUM_EPOCHS
# )

## 11. Final test evaluation
Use the test set only after model selection and training decisions are complete.

In [ ]:
def evaluate_model(model, test_loader, class_names):
    true_labels, predictions = get_predictions(model, test_loader)
    accuracy = accuracy_score(true_labels, predictions)
    macro_f1 = f1_score(
        true_labels, predictions, average="macro", zero_division=0
    )

    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test Macro F1: {macro_f1:.4f}")
    print(classification_report(
        true_labels,
        predictions,
        labels=list(range(len(class_names))),
        target_names=class_names,
        zero_division=0
    ))
    return accuracy, macro_f1

# Uncomment after training and defining the test loaders:
# print("MobileNetV2 Results")
# evaluate_model(model_mnv2, test_loader_mnv2, CLASS_NAMES)
# print("EfficientNet-B0 Results")
# evaluate_model(model_eff, test_loader_eff, CLASS_NAMES)

## Notes before running

- Update `NUM_CLASSES` and `CLASS_NAMES`.
- Define your dataset loaders in Section 6.
- Both models must use the same split assignments and class-index mapping.
- Use each model's pretrained transforms.
- This baseline freezes the backbone; fine-tuning can be added later.
- Checkpoints are selected by validation macro F1; the test set is reserved for final evaluation.